Paso 1: 
- Librerias necesarias
- Definicion de funciones a usar 
- Configuración inicial y conexión a MySQL
Antes de procesar los archivos, necesitamos establecer la conexión a MySQL. Utilizaremos sqlalchemy para la conexión y pandas para la manipulación de datos. scipy.spatial.distance para el calculo de la distacia entre estaciones. json para extraer los datos del archivo obtnenido desde la API de open data bcn. request para extraer esos datos via API.

In [2]:
import pandas as pd
import numpy as np
import requests
import json
import os
import kagglehub
from scipy.spatial.distance import cdist
from sqlalchemy import create_engine, text


In [3]:
# 1. Parámetros de conexión a MySQL
user = 'root'          
password = '12345'   
host = 'localhost'     
database = 'bicing_project'   

# 2. Crear el motor de conexión
engine = create_engine(f'mysql+pymysql://{user}:{password}@{host}/{database}')

# 3. Probar la conexión
with engine.connect() as conn:
    result = conn.execute(text("SELECT 1"))
    print("Conexión exitosa:", result.fetchone())

Conexión exitosa: (1,)


Paso 2: Carga del archivo Resum per mes des de 2009.xlsx
Este archivo tiene varias hojas. Analicemos su contenido:

Hoja "Usos mes": Contiene datos anuales y mensuales de usos totales desde 2009 hasta 2026 (con algunos años con desglose mecánica/eléctrica). También incluye filas con información adicional sobre bicicletas medias anuales y a 31 de diciembre.

Hoja "Abonats": Número de abonados por mes desde 2009 hasta 2025.

Hoja "grafica total usos": Totales anuales de usos (posiblemente un resumen).

Hoja "grafica M E usos": Desglose anual de usos mecánicos y eléctricos, y otros datos como bicicletas medias y rotación.

Hoja "Redha email fecha17523": Datos históricos de bicicletas, estaciones, usos y abonados desde 2007 hasta 2024, además de información de docks.

Para un proyecto bien estructurado, crearemos tablas separadas para cada tipo de información. Empezaremos con la hoja "Usos mes", que es la más completa.

2.1. Leer la hoja "Usos mes"
Observando el archivo, las filas iniciales tienen algunas filas vacías o de encabezado. 
Vamos a leerlo con pandas saltando las filas innecesarias y seleccionando el rango correcto.

In [4]:
# 1. Leer la hoja "Usos mes"
usos_mes= r"C:\Users\ramir\OneDrive\Escritorio\Ramiro\Cursos IT Academy\Bootcamp-Analisis-de-datos\Proyecto\Resum per mes des de 2009.xlsx"

df_usos_mes = pd.read_excel(usos_mes, sheet_name='Usos mes', header=None)

# 2. Mostrar las primeras filas para identificar la estructura
print(df_usos_mes.head(10))

     0        1        2        3        4        5        6        7   \
0   NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN   
1   NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN   
2   NaN      NaN      NaN      NaN      NaN      NaN      NaN      NaN   
3   Any    Gener   Febrer     Març    Abril     Maig     Juny   Juliol   
4  2009   804327   805976   983955   921958  1090461  1063086   996441   
5  2010   650434   682235   845730   940749  1017494  1107099  1078969   
6  2011   938368  1004485  1092630  1187003  1418402  1307452  1314539   
7  2012  1253361  1187905  1473740  1275801  1628865  1585696  1441845   
8  2013  1160485  1034615  1088179  1220726  1333528  1341721  1350829   
9  2014  1043927  1053178  1186481  1132421  1244309  1242143  1214974   

        8         9        10         11        12        13  14   15   16  \
0      NaN       NaN      NaN        NaN       NaN       NaN NaN  NaN  NaN   
1      NaN       NaN      NaN

2.2. DataFrame df_usos
Basado en el contenido que vemos en el archivo, la estructura es:
Las primeras filas (0,1,2) están vacías.
La fila 3 (índice 3) contiene los nombres de columna: Any, Gener, Febrer, ..., Desembre, Total.
Luego vienen los datos desde la fila 4 (año 2009) hasta la fila 20 (año 2025).
Además, hay filas más abajo con información de uso filtrado por mecanicas y electricas, y de bicicletas medias y a 31 de diciembre, pero esas las trataremos por separado.
Entonces, para los usos mensuales, tomaremos las filas 4 a 21 (años 2009-2025) y luego la fila 21 para 2026.

In [5]:
# 1. Dataframe auxiliar para obtener el valor de M56
df_aux = pd.read_excel(usos_mes, sheet_name='Usos mes', header=None, 
                       skiprows=55, nrows=1, usecols=[12])
valor_m56 = df_aux.iloc[0, 0]

# 2. Saltar las primeras 3 filas y usar la fila 3 como cabecera
df_usos = pd.read_excel(usos_mes, sheet_name='Usos mes', header=3, nrows=18)

# 3. Asignar valor faltante de diciembre 2025 al DataFrame principal en la posición de M21
df_usos.at[16, 'Desembre'] = valor_m56

# 4. Eliminar columnas vacías o irrelevantes (por ejemplo, las columnas después de 'Desembre' que pueden estar vacías)
df_usos = df_usos.loc[:, ~df_usos.columns.str.contains('^Unnamed')]

# 5. Eliminar espacios en blanco en nombres de columnas
df_usos.columns = df_usos.columns.str.strip()

# 6. Renombrar columnas para que sean más manejables
df_usos.rename(columns={'Any': 'año'}, inplace=True)

In [6]:
# 1. Definimos los meses en español
meses_es = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio', 'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']

# 2. Mantenemos la lista original solo para "derretir" el DataFrame correctamente al transformar a formato largo (año, mes, usos)
meses_cat = ['Gener', 'Febrer', 'Març', 'Abril', 'Maig', 'Juny', 'Juliol', 'Agost', 'Setembre', 'Octubre', 'Novembre', 'Desembre']

# 3. Diccionario para mapear de Catalán a Español
cat_a_es = dict(zip(meses_cat, meses_es))

# 4. Diccionario para mapear a número de mes (usando la lista en español)
mes_num = {mes: i+1 for i, mes in enumerate(meses_es)}

In [7]:
# 5. Derretir el DataFrame (pivot longer)
df_usos_long = pd.melt(
    df_usos, id_vars=['año'], 
    value_vars=meses_cat, 
    var_name='mes', 
    value_name='usos'
)

# 6. Traducir la columna 'mes' a español
df_usos_long['mes'] = df_usos_long['mes'].map(cat_a_es)

# 7. Agregar columna de número de mes
df_usos_long['mes_num'] = df_usos_long['mes'].map(mes_num)

# 8. Convertir usos a entero
df_usos_long['usos'] = df_usos_long['usos'].astype(int)

# 9. Reordenar columnas (año, mes, mes_num, usos,)
df_usos_long = df_usos_long[['año', 'mes', 'mes_num', 'usos']]

# 10. Mostrar resultado
df_usos_long

,año,mes,mes_num,usos
0,2009,Enero,1,804327
1,2010,Enero,1,650434
2,2011,Enero,1,938368
3,2012,Enero,1,1253361
4,2013,Enero,1,1160485
...,...,...,...,...
199,2021,Diciembre,12,1012627
200,2022,Diciembre,12,1202468
201,2023,Diciembre,12,1207907
202,2024,Diciembre,12,1343507


2.3 DataFrame df_usos_filt
Datos de usos filtrados por tipo (Electrica-Mecanica)

Metodo "extract"
1. La estructura
df_filt['año_tipo'].str.extract(...) busca un patrón de texto dentro de cada fila de la columna año_tipo. Como el patrón tiene dos grupos (definidos por los paréntesis), Pandas devuelve dos columnas, que guardas automáticamente en año y tipo.
2. El patrón (Regex) explicado
El texto r'(\d{4})\s+(.*)' es una Expresión Regular (Regex). Vamos a leerlo de izquierda a derecha:
r: Indica que es un "raw string" (cadena en crudo). Es una buena práctica para que Python no confunda las barras invertidas \ con comandos especiales.
(\d{4}) (Primer Grupo):
\d significa "un dígito" (número).
{4} significa "exactamente cuatro veces".
Los paréntesis () crean el primer grupo, que se enviará a la columna año.
Resultado: Busca el año (ej. 2019).
\s+:
\s busca un espacio en blanco (espacio, tabulación, etc.).
+ significa "uno o más". Esto es útil si entre el año y el tipo hay más de un espacio.
(.*) (Segundo Grupo):
. significa "cualquier carácter" (letras, símbolos, espacios).
* significa "cero o más veces" hasta el final de la línea.
Los paréntesis crean el segundo grupo, que se enviará a la columna tipo.
Resultado: Captura todo lo que sobra después del año (ej. "Mecanica").

In [8]:
# 1. Cargar datos
df_usos_raw = pd.read_excel(usos_mes, sheet_name='Usos mes', skiprows=24, nrows=39, header=None)

# 2. Ajustar columnas (forzamos que df_raw solo tenga las columnas que nos interesan)
# Tomamos solo las primeras 14 columnas para evitar el error de "Length mismatch"
df_usos_raw = df_usos_raw.iloc[:, :14] 
columnas = ['año_tipo'] + meses_es + ['total']
df_usos_raw.columns = columnas

# 3. Limpiar filas irrelevantes
# Primero, eliminamos las filas donde la primera columna sea NaN (las filas de totales vacías)
df_usos_filt = df_usos_raw.dropna(subset=['año_tipo'])
#df_usos_filt = df_usos_raw[df_usos_raw['año_tipo'].notna()].copy()
#df_usos_filt = df_usos_raw[~df_usos_raw['año_tipo'].str.contains('Total|Dif|%|informacion|COMET', case=False, na=False)]
# Luego, eliminamos las filas de "Dif", "Informacion", etc.
df_usos_filt = df_usos_filt[~df_usos_filt['año_tipo'].astype(str).str.contains('Dif|%|informacion|COMET', case=False, na=False)]

# 4. Separar el Año y el Tipo
df_usos_filt[['año', 'tipo']] = df_usos_filt['año_tipo'].str.extract(r'(\d{4})\s+(.*)')

# 5. Transformar a formato largo (Melt)
# IMPORTANTE: Usamos meses_es porque ya renombramos las columnas arriba
df_usos_filt_long = pd.melt(
    df_usos_filt, 
    id_vars=['año', 'tipo'], 
    value_vars=meses_es, 
    var_name='mes', 
    value_name='usos'
)

# 6. Agregar número de mes (usando el diccionario que ya tenías de meses_es)
df_usos_filt_long['mes_num'] = df_usos_filt_long['mes'].map(mes_num)

# Convertir usos a numérico y limpiar
df_usos_filt_long['usos'] = pd.to_numeric(df_usos_filt_long['usos'], errors='coerce')
df_usos_filt_long = df_usos_filt_long.dropna(subset=['usos'])
df_usos_filt_long['usos'] = df_usos_filt_long['usos'].astype(int)

# 7. Reordenar columnas (usando los nombres correctos: año, mes, mes_num, usos, tipo)
df_usos_filt_long = df_usos_filt_long[['año', 'mes', 'mes_num', 'usos', 'tipo']]

df_usos_filt_long


,año,mes,mes_num,usos,tipo
0,2019,Enero,1,842435,Mecanica
1,2019,Enero,1,20,Electrica
2,2020,Enero,1,794037,Mecanica
3,2020,Enero,1,240650,Electrica
4,2021,Enero,1,653456,Mecanica
...,...,...,...,...,...
187,2024,Diciembre,12,1109106,Electrica
188,2025,Diciembre,12,193089,Mecanica
189,2025,Diciembre,12,1176207,Electrica
190,2026,Diciembre,12,0,Mecanica


2.4. DataFrame df_inv_bici Datos de inventario de la flota disponible en el sistema.

1. Bicicletes mitja any (Bicicletas media año)
Este valor representa el promedio de bicicletas disponibles para el usuario a lo largo de todo el año. 
Es un indicador de la capacidad operativa real del sistema.
Se calcula promediando el número de unidades que han estado activas cada día. Por ejemplo, en la fila de 2022 (Fila 38), la media fue de 4.047 mecánicas y 2.953 eléctricas.

2. Bicicletes a 31/12 (Bicicletas a 31 de diciembre)
Es un dato de "foto fija" o inventario final. Indica el número exacto de bicicletas que formaban parte de la flota el último día del año. 
Sirve para ver el crecimiento real de la infraestructura de un año a otro.
En la misma fila de 2022, el año cerró con un total de 7.000 unidades (4.000 mecánicas y 3.000 eléctricas).

Este dato es fundamental para calcular el ratio de rotación: si dividimos el "Total de Usos" (Columna N) por la "Bicicleta media" (Columna P o Q), obtendremos cuántas veces se usó cada bicicleta de media ese año.


In [9]:
# 1. Cargar el rango: Columna B (años) y P-S (inventario)
# Usamos usecols para saltar el bloque de meses central
df_inv_raw = pd.read_excel(usos_mes, sheet_name='Usos mes', skiprows=24, nrows=39, usecols="A, P:S", header=None)

# 2. Asignar nombres de columnas
# P=mitja_mec, Q=mitja_elec, R=final_mec, S=final_elec
df_inv_raw.columns = ['año_tipo', 'mitja_mec', 'mitja_elec', 'final_mec', 'final_elec']

# 3. Limpieza de filas irrelevantes
#df_inv = df_inv_raw[df_inv_raw['año_tipo'].notna()].copy()
# Filtramos para quedarnos solo con las filas que tienen "Mecanica" o "Electrica"
df_inv = df_inv_raw[df_inv_raw['año_tipo'].str.contains('Mecanica', case=False, na=False)]

# 4. Extraer Año y Tipo (tu función Regex favorita)
#df_inv[['año', 'tipo']] = df_inv['año_tipo'].str.extract(r'(\d{4})\s+(.*)')
df_inv['año'] = df_inv['año_tipo'].str.extract(r'(\d{4})')

# 5. Convertir a numérico 
# Las celdas vacías de 2025/2026 se convertirán automáticamente en NaN (Not a Number)
cols_num = ['mitja_mec', 'mitja_elec', 'final_mec', 'final_elec']
for col in cols_num:
    df_inv[col] = pd.to_numeric(df_inv[col], errors='coerce')

# 6. Renombrar y ordenar columnas finales
df_inventario = df_inv[['año', 'mitja_mec', 'mitja_elec', 'final_mec', 'final_elec']]
df_inventario = df_inventario.rename(columns={'mitja_mec': 'media_mec', 'mitja_elec': 'media_elec'})

# Ver el resultado (incluyendo los vacíos de 2025)
df_inventario


C:\Users\ramir\AppData\Local\Temp\ipykernel_29184\2842951775.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_inv['año'] = df_inv['año_tipo'].str.extract(r'(\d{4})')
C:\Users\ramir\AppData\Local\Temp\ipykernel_29184\2842951775.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_inv[col] = pd.to_numeric(df_inv[col], errors='coerce')


,año,media_mec,media_elec,final_mec,final_elec
0,2019,6160.0,840.0,6150.0,850.0
4,2020,5568.0,1432.0,5076.0,1924.0
8,2021,4900.0,2100.0,4397.0,2603.0
13,2022,4047.0,2953.0,4000.0,3000.0
18,2023,3487.0,3530.0,3108.0,4000.0
23,2024,3085.0,4198.0,3000.0,4608.0
29,2025,NaN,NaN,NaN,NaN
36,2026,NaN,NaN,NaN,NaN


In [10]:
# 1. Carga inicial dataframe de abonados 
df_abonados_raw = pd.read_excel(usos_mes, sheet_name='Abonats', skiprows=3, nrows=17)
df_abonados_raw = df_abonados_raw.iloc[:, 0:13]  # Seleccionamos 'Any' y los 12 meses

# --- PASO CLAVE: Limpiar espacios en blanco en los nombres de las columnas ---
df_abonados_raw.columns = df_abonados_raw.columns.str.strip()

# 2. Transformar a formato largo (Long)
# Usamos meses_cat que ya definiste para que Pandas sepa qué columnas "derretir"
df_abonados_long = df_abonados_raw.melt(
    id_vars=['Any'], 
    value_vars=meses_cat, 
    var_name='mes', 
    value_name='abonados'
)

# 3. Traducir los meses a español usando tu diccionario cat_a_es
df_abonados_long['mes'] = df_abonados_long['mes'].map(cat_a_es)

# 4. Añadir el número de mes (basándonos en la lista meses_es que ya tienes)
df_abonados_long['mes_num'] = df_abonados_long['mes'].map(mes_num)

# 5. Renombrar columnas finales y ordenar
df_abonados_long = df_abonados_long.rename(columns={'Any': 'año'})
df_abonados_long = df_abonados_long.sort_values(['año', 'mes_num']).reset_index(drop=True)

# Reordenar columnas para que sea más legible
df_abonados_long = df_abonados_long[['año', 'mes', 'mes_num', 'abonados']]
df_abonados_long


,año,mes,mes_num,abonados
0,2009,Enero,1,133117.0
1,2009,Febrero,2,129601.0
2,2009,Marzo,3,127225.0
3,2009,Abril,4,124477.0
4,2009,Mayo,5,119546.0
...,...,...,...,...
199,2025,Agosto,8,164927.0
200,2025,Septiembre,9,166001.0
201,2025,Octubre,10,166510.0
202,2025,Noviembre,11,166488.0


3. Dataset Información de las estaciones del nuevo Bicing de la ciudad de Barcelona open data bcn
Vamos a trabajar con el archivo Informacio_Estacions_Bicing_securitzat.json que descargamos. Primero, entenderemos qué contiene y luego lo convertiremos a un DataFrame de pandas.

3.1.: Entender la estructura del JSON
Este archivo contiene información detallada de todas las estaciones de Bicing en la actualidad. Es un archivo de texto con formato JSON (JavaScript Object Notation), que es una forma estándar de intercambiar datos.

Para trabajar con él en Python, necesitaremos abrirlo, leer su contenido y convertirlo en una estructura de datos de Python (como diccionarios y listas). Luego, extraeremos la información relevante para crear un DataFrame.

3.2.: Leer el archivo JSON
Usaremos la librería json de Python para cargar el archivo.

In [11]:
# 1. Datos necesarios
url = "https://opendata-ajuntament.barcelona.cat/data/dataset/bd2462df-6e1e-4e37-8205-a4b8e7313b84/resource/f60e9291-5aaa-417d-9b91-612a9de800aa/download"
mi_token = "4d298cfce32e59d76e4e2f38c0a05fcd021cc5d5411a573e3c680cc51c7b5fd1"
ruta_guardado = r"C:\Users\ramir\OneDrive\Escritorio\Ramiro\Cursos IT Academy\Bootcamp-Analisis-de-datos\Proyecto\informacion_estaciones_bicing.json"

# 2. Descargar los datos usando el token
respuesta = requests.get(url, headers={"Authorization": mi_token})

# 3. Convertir a JSON y guardar el archivo
data = respuesta.json()

with open(ruta_guardado, 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=4)

print("Archivo descargado y guardado correctamente.")


Archivo descargado y guardado correctamente.


In [12]:
# 4. Visualizar el contenido del JSON para entender su estructura
data

{'last_updated': 1773257866,
 'ttl': 0,
 'data': {'stations': [{'station_id': 1,
    'name': 'GRAN VIA CORTS CATALANES, 760',
    'physical_configuration': 'ELECTRICBIKESTATION',
    'lat': 41.3979779,
    'lon': 2.1801069,
    'altitude': 16.0,
    'address': 'GRAN VIA CORTS CATALANES, 760',
    'cross_street': '02-Eixample/05-el Fort Pienc',
    'post_code': '08013',
    'capacity': 46,
    'is_charging_station': True,
    'short_name': 1,
    'nearby_distance': 1000.0,
    '_ride_code_support': True,
    'rental_uris': None},
   {'station_id': 2,
    'name': 'C/ ROGER DE FLOR, 126',
    'physical_configuration': 'ELECTRICBIKESTATION',
    'lat': 41.3954877,
    'lon': 2.1771985,
    'altitude': 17.0,
    'address': 'C/ ROGER DE FLOR, 126',
    'cross_street': '02-Eixample/05-el Fort Pienc',
    'post_code': '08013',
    'capacity': 29,
    'is_charging_station': True,
    'short_name': 2,
    'nearby_distance': 1000.0,
    '_ride_code_support': True,
    'rental_uris': None},
   {'s

3.3.: Convertir los datos a un dataframe
Ya tenemos los datos reales. Lo que vemos dentro de "data" es la estructura estándar de las APIs de transporte (llamada GBFS). Los datos que nos interesan están "escondidos" dentro de las llaves data y luego stations. 
Para trabajar con esto de forma fácil (como una tabla), lo mejor es convertir esa lista de estaciones en un DataFrame de Pandas.

Proceso:
- data['data']['stations']: Entramos al primer nivel (data) y luego al segundo (stations) donde vive la lista de diccionarios.
- pd.DataFrame(...): Pandas es inteligente y transforma esa lista de diccionarios directamente en una tabla donde cada clave (como name, lat, capacity) es una columna.


In [13]:
# 1. Accedemos a la lista de estaciones que está dentro del diccionario 'data'
lista_estaciones = data['data']['stations']

# 2. Lo convertimos en una tabla (DataFrame)
df_estaciones = pd.DataFrame(lista_estaciones)

# 3. Visualizamos las primeras filas
df_estaciones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 544 entries, 0 to 543
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   station_id              544 non-null    int64  
 1   name                    544 non-null    object 
 2   physical_configuration  544 non-null    object 
 3   lat                     544 non-null    float64
 4   lon                     544 non-null    float64
 5   altitude                519 non-null    float64
 6   address                 544 non-null    object 
 7   cross_street            542 non-null    object 
 8   post_code               540 non-null    object 
 9   capacity                544 non-null    int64  
 10  is_charging_station     544 non-null    bool   
 11  short_name              544 non-null    int64  
 12  nearby_distance         544 non-null    float64
 13  _ride_code_support      544 non-null    bool   
 14  rental_uris             0 non-null      ob

3.4 Limpiamos el DataFrame
- Observamos que existen estaciones que tienen valor NaN en altitude. Filtramos esas 23 estaciones y por medio de una API gratuita "https://api.open-elevation.com/api/v1/lookup" (https://github.com/Jorl17/open-elevation/blob/master/docs/api.md) obtenemos su altitud.
- Observamos tambien que en la columna "nearby_distance" todos los valores dicen 1000.0, lo cual no es un dato real. Por parte de la compania de Bicing nos dijeron que las estaciones estan distribuidas mas o menos entre 200 y 300 metros de distancia unas de otras (en promedio). 1000.0 es un parámetro por defecto del sistema GBFS que indica el radio máximo de búsqueda de estaciones cercanas para un usuario, no la distancia real entre ellas. Para obtener la distancia real a la estación más cercana, debemos calcularla matemáticamente usando las coordenadas de latitud y longitud. La forma más precisa es mediante la fórmula de Haversine (que considera la curvatura de la Tierra). 
- Finalmente limpiamos el DataFrame y nos quedamos con las columnas que nos quedamos con las columnas que aportan informacion util, y las que no como por ejemplo: 
    - physical_configuration: Casi todas son ELECTRICBIKESTATION.
    - rental_uris y _ride_code_support: Son datos técnicos de la app que no sirven para análisis de datos.
    - nearby_distance: Suele ser el mismo valor para todas (1000m).
    - is_valet_station: Está lleno de NaN (valores nulos).
    - nearby_distance: valor 1000.0 por defecto
Las eliminamos del dataframe final

In [14]:
# 1. Filtramos las 23 estaciones que no tienen altitud
estaciones_sin_altitud = df_estaciones[df_estaciones['altitude'].isna()].copy()

# 2. Preparamos el formato JSON que pide la API (lista de diccionarios)
locations = []
for index, row in estaciones_sin_altitud.iterrows():
    locations.append({"latitude": row['lat'], "longitude": row['lon']})

# 3. Hacemos una única petición POST con todas las ubicaciones
url = "https://api.open-elevation.com/api/v1/lookup"
payload = {"locations": locations}

try:
    response = requests.post(url, json=payload, timeout=15)
    data_respuesta = response.json()
    
    # 4. Extraemos las elevaciones y las guardamos en el DataFrame original
    # La API devuelve los resultados en el mismo orden que los enviamos
    elevaciones = [res['elevation'] for res in data_respuesta['results']]
    
    # Asignamos los valores exactos a las filas correspondientes
    df_estaciones.loc[df_estaciones['altitude'].isna(), 'altitude'] = elevaciones
    
    print(f"Se han actualizado {len(elevaciones)} altitudes.")

except Exception as e:
    print(f"Error al conectar con la API: {e}")

# 5. Verificamos que ya no queden nulos
print(f"Valores nulos restantes: {df_estaciones['altitude'].isna().sum()}")


Se han actualizado 25 altitudes.
Valores nulos restantes: 0


In [15]:
# Definimos la función Haversine para que cdist la pueda usar para calcular las distancias entre coordenadas geográficas
def haversine(u, v):
    r = 6371000 # Radio de la Tierra en metros
    lat1, lon1 = u
    lat2, lon2 = v
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * r * np.arcsin(np.sqrt(a))

In [16]:
# 1. Extraemos las coordenadas (lat, lon) y las convertimos a radianes
coords_rad = np.radians(df_estaciones[['lat', 'lon']].values)

# 2. Calculamos la matriz de distancias usando nuestra función
# cdist ahora usará la fórmula matemática para cada par de puntos
dist_matrix = cdist(coords_rad, coords_rad, metric=haversine)

# 3. Ponemos la diagonal a infinito (para que una estación no sea su propia vecina más cercana)
np.fill_diagonal(dist_matrix, np.inf)

# 4. Buscamos la distancia al vecino más cercano (ahora ya está en metros)
distancia_minima_metros = np.min(dist_matrix, axis=1)

# 5. Guardamos el resultado en el DataFrame
df_estaciones['nearby_distance_real'] = distancia_minima_metros.round(2)

# 6. Revisamos el promedio para ver si coincide con lo que te dijeron (200-300m)
print(f"Distancia promedio real: {df_estaciones['nearby_distance_real'].mean():.2f} metros")
df_estaciones.head(2)


Distancia promedio real: 205.47 metros


,station_id,name,physical_configuration,lat,lon,altitude,address,cross_street,post_code,capacity,is_charging_station,short_name,nearby_distance,_ride_code_support,rental_uris,is_valet_station,nearby_distance_real
0,1,"GRAN VIA CORTS CATALANES, 760",ELECTRICBIKESTATION,41.397978,2.180107,16.0,"GRAN VIA CORTS CATALANES, 760",02-Eixample/05-el Fort Pienc,08013,46,True,1,1000.0,True,None,NaN,244.52
1,2,"C/ ROGER DE FLOR, 126",ELECTRICBIKESTATION,41.395488,2.177198,17.0,"C/ ROGER DE FLOR, 126",02-Eixample/05-el Fort Pienc,08013,29,True,2,1000.0,True,None,NaN,130.03


In [19]:
# 1. Seleccionamos solo las columnas realmente útiles
columnas_interes = [
    'station_id', 'name', 'lat', 'lon', 'altitude', 
    'address', 'cross_street', 'post_code', 'capacity', 'is_charging_station', 'short_name', 'nearby_distance_real'
]

# 2. Creamos el nuevo DataFrame filtrado
df_estaciones_limpio = df_estaciones[columnas_interes]

# 3. Ver el resultado final
print(df_estaciones_limpio.info()) # Para ver si hay nulos
df_estaciones_limpio.head(20)    


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 544 entries, 0 to 543
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   station_id            544 non-null    int64  
 1   name                  544 non-null    object 
 2   lat                   544 non-null    float64
 3   lon                   544 non-null    float64
 4   altitude              544 non-null    float64
 5   address               544 non-null    object 
 6   cross_street          542 non-null    object 
 7   post_code             540 non-null    object 
 8   capacity              544 non-null    int64  
 9   is_charging_station   544 non-null    bool   
 10  short_name            544 non-null    int64  
 11  nearby_distance_real  544 non-null    float64
dtypes: bool(1), float64(4), int64(3), object(4)
memory usage: 47.4+ KB
None


,station_id,name,lat,lon,altitude,address,cross_street,post_code,capacity,is_charging_station,short_name,nearby_distance_real
0,1,"GRAN VIA CORTS CATALANES, 760",41.397978,2.180107,16.0,"GRAN VIA CORTS CATALANES, 760",02-Eixample/05-el Fort Pienc,08013,46,True,1,244.52
1,2,"C/ ROGER DE FLOR, 126",41.395488,2.177198,17.0,"C/ ROGER DE FLOR, 126",02-Eixample/05-el Fort Pienc,08013,29,True,2,130.03
2,3,"C/ NÀPOLS, 82",41.394156,2.181331,11.0,"C/ NÀPOLS, 82",02-Eixample/05-el Fort Pienc,08013,27,True,3,93.48
3,4,"C/ RIBES, 13",41.393317,2.181248,8.0,"C/ RIBES, 13",02-Eixample/05-el Fort Pienc,08013,21,True,4,93.48
4,5,"PG. LLUIS COMPANYS, 11 (ARC TRIOMF)",41.391103,2.180176,7.0,"PG. LLUIS COMPANYS, 11 (ARC TRIOMF)","01-CiutatVella/04-Sant Pere, Santa Caterina i ...",08018,39,True,5,6.94
5,6,"CETT-PG. LLUIS COMPANYS, 18 (ARC TRIOMF)",41.391429,2.180569,10.0,"CETT-PG. LLUIS COMPANYS, 18 (ARC TRIOMF)","01-CiutatVella/04-Sant Pere, Santa Caterina i ...",08018,39,True,6,26.01
6,7,"PG. PUJADES, 1 (JUTJATS)",41.388885,2.183290,6.0,"PG. PUJADES, 1 (JUTJATS)","01-CiutatVella/04-Sant Pere, Santa Caterina i ...",08003,27,True,7,32.38
7,8,"PG. PUJADES, 2",41.389135,2.183489,6.0,"PG. PUJADES, 2","01-CiutatVella/04-Sant Pere, Santa Caterina i ...",08003,27,True,8,32.38
8,9,"AV. MARQUÉS DE L'ARGENTERA,13",41.384546,2.184922,5.0,"AV. MARQUÉS DE L'ARGENTERA,13","01-CiutatVella/04-Sant Pere, Santa Caterina i ...",08003,27,True,9,35.82
9,10,"C/ 60, NÚMERO 25",41.346775,2.143623,15.0,"C/ 60, NÚMERO 25",03-Sants-Montjuïc/12-la Marina del Prat Vermell,08040,43,True,10,520.55


In [18]:
# 1. Definimos tu ruta personalizada
#mi_ruta = r"C:\Users\ramir\OneDrive\Escritorio\Ramiro\Cursos IT Academy\Bootcamp-Analisis-de-datos\Proyecto"

# 2. Configuramos la variable de entorno para que kagglehub use esa carpeta
#os.environ["KAGGLEHUB_CACHE"] = mi_ruta

# 3. Descargamos el dataset
# Nota: kagglehub creará subcarpetas dentro de tu ruta (datasets/edomingo/...)
#path = kagglehub.dataset_download("edomingo/bicing-stations-dataset-bcn-bike-sharing")

#print("Los archivos están listos en:", path)
